# DSA 4060XA – Week 6 Guided Evaluation Lab
**Name:** Nicholas Kinyanjui  
**Student ID:** 670178  
**Repository:** `DSA4060XA-Week6-670178`

Compare a popularity baseline, genre content model, and item-based collaborative filtering
using rating accuracy, ranking quality, catalogue coverage, novelty, and diversity.

**Source completeness:** The supplied instructions start midway through Step 11.
Steps 1–11 below are a documented reconstruction, not a claim to reproduce missing code.
Assumptions: deterministic random 80/20 split within each user (seed 42), binary genre
features, cosine similarity, rating-weighted content predictions, and raw-rating item CF.
The observed rating scale is 2–5; predictions are bounded to the assumed 1–5 scale.
If the missing pages specify other methods, revise those sections before submission.

All fitted statistics and collaborative similarities use training ratings only.
Movie genres are catalogue metadata. Test ratings are used only for evaluation.

## Steps 1–3 – Load, inspect, and validate
Keep both CSV files beside this notebook. Only NumPy and pandas are needed for calculations.
The cosine function below avoids an additional machine-learning dependency.

In [1]:
from pathlib import Path
from itertools import combinations
import platform
import numpy as np
import pandas as pd
from IPython.display import display

SEED = 42
TOP_K = 3
RELEVANCE_THRESHOLD = 4
DATA_DIR = Path.cwd()
movies = pd.read_csv(DATA_DIR / "DSA4060_Week6_Movies.csv")
ratings = pd.read_csv(DATA_DIR / "DSA4060_Week6_Ratings.csv")

assert set(movies.columns) == {"movie_id", "title", "genres"}
assert set(ratings.columns) == {"user_id", "movie_id", "rating", "timestamp"}
assert not movies.isna().any().any() and not ratings.isna().any().any()
assert movies["movie_id"].is_unique
assert not ratings.duplicated(["user_id", "movie_id"]).any()
assert ratings["movie_id"].isin(movies["movie_id"]).all()
assert ratings["rating"].between(1, 5).all()
print(f"Python {platform.python_version()} | NumPy {np.__version__} | pandas {pd.__version__}")
print(f"{len(movies)} movies; {ratings.user_id.nunique()} users; {len(ratings)} ratings")
print(f"Observed rating range: {ratings.rating.min()}–{ratings.rating.max()}")
display(movies.head(), ratings.head())
display(ratings.groupby("user_id").agg(ratings=("rating", "size"), mean_rating=("rating", "mean")))

Python 3.12.14 | NumPy 2.3.5 | pandas 3.0.1
20 movies; 12 users; 120 ratings
Observed rating range: 2–5


,movie_id,title,genres
0,1,Galaxy Mission,Action|Sci-Fi
1,2,Love in Nairobi,Romance|Drama
2,3,Final Battle,Action
3,4,Family Weekend,Comedy|Family
4,5,Future World,Sci-Fi|Action


,user_id,movie_id,rating,timestamp
0,1,2,3,1
1,1,10,3,2
2,1,1,5,3
3,1,9,3,4
4,1,7,3,5


,ratings,mean_rating
user_id,,
1,10,3.2
2,10,3.5
3,10,3.3
4,10,2.9
5,10,2.9
6,10,3.6
7,10,3.3
8,10,3.0
9,10,3.1


## Step 4 – Per-user train/test split
Hold out two of each user's ten ratings using seed 42, leaving eight for training.
This evaluates known users with partially observed histories. `timestamp` runs from
1 to 120 and is treated as an ordering field, not a verified calendar timestamp.
The random split does not test chronological performance.

In [2]:
rng = np.random.default_rng(SEED)
held_out = []
for user_id, group in ratings.sort_values(["user_id", "timestamp"]).groupby("user_id", sort=True):
    n_test = max(1, int(np.ceil(0.2 * len(group))))
    assert n_test < len(group), "Each user needs training history"
    held_out.extend(rng.choice(group.index.to_numpy(), size=n_test, replace=False).tolist())
test = ratings.loc[held_out].sort_values(["user_id", "timestamp"]).reset_index(drop=True)
train = ratings.drop(index=held_out).sort_values(["user_id", "timestamp"]).reset_index(drop=True)
assert len(train) + len(test) == len(ratings)
assert set(map(tuple, train[["user_id", "movie_id"]].to_numpy())).isdisjoint(
    set(map(tuple, test[["user_id", "movie_id"]].to_numpy())))
assert set(train.user_id) == set(test.user_id)
display(pd.DataFrame({"Train": train.groupby("user_id").size(), "Test": test.groupby("user_id").size()}))
print(f"Training ratings: {len(train)}; test ratings: {len(test)}")

,Train,Test
user_id,,
1,8,2
2,8,2
3,8,2
4,8,2
5,8,2
6,8,2
7,8,2
8,8,2
9,8,2


Training ratings: 96; test ratings: 24


## Steps 5–6 – Popularity baseline and rating metrics
Predict each movie's training mean rating, falling back to the global training mean
for unseen movies. Rank by mean rating, then training count, then movie ID for stable ties.
MAE measures average absolute error. RMSE gives larger errors greater influence.

In [3]:
global_mean = float(train.rating.mean())
movie_stats = (movies[["movie_id"]]
    .merge(train.groupby("movie_id").agg(mean_rating=("rating", "mean"),
           rating_count=("rating", "size")).reset_index(), on="movie_id", how="left"))
movie_stats["mean_rating"] = movie_stats["mean_rating"].fillna(global_mean)
movie_stats["rating_count"] = movie_stats["rating_count"].fillna(0).astype(int)
movie_means = movie_stats.set_index("movie_id")["mean_rating"].to_dict()

def baseline_predict(user_id, movie_id):
    return float(movie_means.get(movie_id, global_mean))

def rating_metrics(actual, predicted):
    errors = np.asarray(actual, dtype=float) - np.asarray(predicted, dtype=float)
    return float(np.mean(np.abs(errors))), float(np.sqrt(np.mean(errors ** 2)))

baseline_test = test.copy()
baseline_test["prediction"] = [baseline_predict(u, m) for u, m in test[["user_id", "movie_id"]].itertuples(index=False, name=None)]
baseline_mae, baseline_rmse = rating_metrics(baseline_test.rating, baseline_test.prediction)
display(movie_stats.merge(movies[["movie_id", "title"]], on="movie_id").sort_values(
    ["mean_rating", "rating_count", "movie_id"], ascending=[False, False, True]))
print(f"Baseline MAE: {baseline_mae:.3f}; RMSE: {baseline_rmse:.3f}")

,movie_id,mean_rating,rating_count,title
5,6,4.166667,6,Broken Hearts
0,1,4.000000,3,Galaxy Mission
3,4,4.000000,3,Family Weekend
9,10,3.666667,6,Hidden Truth
6,7,3.500000,8,City Detectives
8,9,3.500000,4,Wedding Weekend
12,13,3.500000,4,Digital Dreams
19,20,3.333333,6,Weekend Escape
11,12,3.333333,3,The Last Case
17,18,3.333333,3,The Promise


Baseline MAE: 0.752; RMSE: 0.920


## Steps 7–9 – Content-based model
Split pipe-separated genres into binary features. Cosine compares the genres of a
candidate with each movie the user rated in training. Predict a similarity-weighted
average of those ratings. If all similarities are zero, use the baseline prediction.
This is a lightweight metadata content model; it does not use reviews or plot text.

In [4]:
item_feature_matrix = movies.set_index("movie_id")["genres"].str.get_dummies(sep="|").astype(float)

def cosine_similarity(matrix):
    values = np.asarray(matrix, dtype=float)
    norms = np.linalg.norm(values, axis=1, keepdims=True)
    normalized = np.divide(values, norms, out=np.zeros_like(values), where=norms > 0)
    return normalized @ normalized.T

genre_similarity = pd.DataFrame(cosine_similarity(item_feature_matrix),
    index=item_feature_matrix.index, columns=item_feature_matrix.index)

def content_score(user_id, movie_id, training_data):
    history = training_data.loc[training_data.user_id == user_id]
    if history.empty:
        return baseline_predict(user_id, movie_id)
    weights = genre_similarity.loc[movie_id, history.movie_id].to_numpy()
    if weights.sum() <= 0:
        return baseline_predict(user_id, movie_id)
    return float(np.clip(np.dot(weights, history.rating.to_numpy()) / weights.sum(), 1, 5))

content_test = test.copy()
content_test["prediction"] = [content_score(u, m, train) for u, m in test[["user_id", "movie_id"]].itertuples(index=False, name=None)]
content_mae, content_rmse = rating_metrics(content_test.rating, content_test.prediction)
display(item_feature_matrix.head())
print(f"Content MAE: {content_mae:.3f}; RMSE: {content_rmse:.3f}")

,Action,Adventure,Comedy,Crime,Drama,Family,Mystery,Romance,Sci-Fi
movie_id,,,,,,,,,
1,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
3,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
5,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


Content MAE: 0.740; RMSE: 0.850


## Steps 10–11 – Item-based collaborative filtering
Build the user–movie matrix from training ratings, filling missing entries with zero
only for computing item cosine similarities. Zero is not an actual rating.
Predict using the user's observed training ratings weighted by item similarity.
Use all positive-similarity neighbors; fall back to the baseline when none exist.
Raw-rating cosine on a small matrix may reflect overlap/popularity as well as taste.

In [5]:
user_item_matrix = (train.pivot(index="user_id", columns="movie_id", values="rating")
    .reindex(columns=movies.movie_id).fillna(0.0))
item_cosines = cosine_similarity(user_item_matrix.T)
np.fill_diagonal(item_cosines, 0.0)
item_similarity = pd.DataFrame(item_cosines,
    index=user_item_matrix.columns, columns=user_item_matrix.columns)

def item_cf_predict(user_id, movie_id, training_data):
    history = training_data.loc[(training_data.user_id == user_id) & (training_data.movie_id != movie_id)]
    if history.empty:
        return baseline_predict(user_id, movie_id)
    weights = item_similarity.loc[movie_id, history.movie_id].to_numpy()
    if weights.sum() <= 0:
        return baseline_predict(user_id, movie_id)
    return float(np.clip(np.dot(weights, history.rating.to_numpy()) / weights.sum(), 1, 5))

cf_test = test.copy()
cf_test["prediction"] = [item_cf_predict(u, m, train) for u, m in test[["user_id", "movie_id"]].itertuples(index=False, name=None)]
cf_mae, cf_rmse = rating_metrics(cf_test.rating, cf_test.prediction)
print(f"CF MAE: {cf_mae:.3f}; RMSE: {cf_rmse:.3f}")

CF MAE: 0.858; RMSE: 1.020


## Step 12 – Compare rating prediction metrics
Lower MAE and RMSE are better. All models predict the identical held-out rows.

In [6]:
rating_results = pd.DataFrame({
    "Model": ["Popularity Baseline", "Content-Based", "Item-Based CF"],
    "MAE": [baseline_mae, content_mae, cf_mae],
    "RMSE": [baseline_rmse, content_rmse, cf_rmse]})
display(rating_results.sort_values("MAE").round(4))
mae_winner = rating_results.loc[rating_results.MAE.idxmin(), "Model"]
rmse_winner = rating_results.loc[rating_results.RMSE.idxmin(), "Model"]
print("6. Lowest MAE:", mae_winner)
print("7. Lowest RMSE:", rmse_winner)
for row in rating_results.iloc[1:].itertuples(index=False):
    print(f"8. {row.Model}: MAE improvement over baseline = {baseline_mae - row.MAE:+.4f}; "
          f"RMSE improvement = {baseline_rmse - row.RMSE:+.4f}")
print("9. MAE and RMSE weigh errors differently. Neither measures list relevance or discovery. "
      "A numerical win on this one small split is not evidence of a statistically clear advantage.")

,Model,MAE,RMSE
1,Content-Based,0.7405,0.8496
0,Popularity Baseline,0.7523,0.9204
2,Item-Based CF,0.8584,1.0195


6. Lowest MAE: Content-Based
7. Lowest RMSE: Content-Based
8. Content-Based: MAE improvement over baseline = +0.0118; RMSE improvement = +0.0708
8. Item-Based CF: MAE improvement over baseline = -0.1061; RMSE improvement = -0.0992
9. MAE and RMSE weigh errors differently. Neither measures list relevance or discovery. A numerical win on this one small split is not evidence of a statistically clear advantage.


## Steps 13–14 – Top-N recommendations
An item is relevant when its held-out rating is at least 4. Candidates are **all catalogue
movies not rated in training**, including held-out and completely unrated movies.
Relevance is unknown for unrated candidates; the lab counts no hit for them.
Ties are broken by movie ID. Filtering seen items makes popularity lists differ across
users, but popularity still applies one shared order rather than learning user taste.

In [7]:
test["relevant"] = test.rating >= RELEVANCE_THRESHOLD
all_movie_ids = sorted(movies.movie_id.tolist())

def unseen_movies(user_id):
    seen = set(train.loc[train.user_id == user_id, "movie_id"])
    return [m for m in all_movie_ids if m not in seen]

popularity_order = movie_stats.sort_values(
    ["mean_rating", "rating_count", "movie_id"], ascending=[False, False, True]).movie_id.tolist()

def recommend_popularity(user_id, k=3):
    unseen = set(unseen_movies(user_id))
    return [m for m in popularity_order if m in unseen][:k]

def recommend_content(user_id, k=3):
    scored = [(m, content_score(user_id, m, train)) for m in unseen_movies(user_id)]
    return [m for m, score in sorted(scored, key=lambda pair: (-pair[1], pair[0]))[:k]]

def recommend_cf(user_id, k=3):
    scored = [(m, item_cf_predict(user_id, m, train)) for m in unseen_movies(user_id)]
    return [m for m, score in sorted(scored, key=lambda pair: (-pair[1], pair[0]))[:k]]

recommenders = {"Popularity Baseline": recommend_popularity,
                "Content-Based": recommend_content, "Item-Based CF": recommend_cf}
title_by_id = movies.set_index("movie_id").title.to_dict()
for name, recommender in recommenders.items():
    print(name, [(m, title_by_id[m]) for m in recommender(1)])
print("Content and CF use individual training histories. Content is easiest to explain "
      "personally through shared genres; popularity is easiest to explain globally.")

Popularity Baseline

 [(6, 'Broken Hearts'), (20, 'Weekend Escape'), (12, 'The Last Case')]
Content-Based [(3, 'Final Battle'), (5, 'Future World'), (8, 'Space Warriors')]
Item-Based CF [(11, 'Laugh Out Loud'), (18, 'The Promise'), (5, 'Future World')]
Content and CF use individual training histories. Content is easiest to explain personally through shared genres; popularity is easiest to explain globally.


## Step 15 – Precision@K, Recall@K, NDCG@K
Precision divides relevant hits by K. Recall divides hits by the number of relevant held-out
items. Binary NDCG discounts lower-ranked hits and normalizes by the ideal ranking.
Following the lab, users with zero relevant held-out items are excluded from all three means.
We report that denominator explicitly; these are macro averages across eligible users.

In [8]:
def precision_recall_at_k(recs, relevant, k):
    hits = len(set(recs[:k]) & set(relevant))
    return hits / k, hits / len(relevant) if relevant else np.nan

def ndcg_at_k(recs, relevant, k):
    dcg = sum(1 / np.log2(rank + 1) for rank, movie_id in enumerate(recs[:k], start=1)
              if movie_id in relevant)
    idcg = sum(1 / np.log2(rank + 1) for rank in range(1, min(len(relevant), k) + 1))
    return dcg / idcg if idcg > 0 else np.nan

def evaluate_ranking(recommender, k=3):
    rows = []
    for user_id in sorted(test.user_id.unique()):
        relevant = test.loc[(test.user_id == user_id) & test.relevant, "movie_id"].tolist()
        if not relevant:
            continue
        recs = recommender(user_id, k)
        precision, recall = precision_recall_at_k(recs, relevant, k)
        rows.append({"user_id": user_id, "Precision": precision, "Recall": recall,
                     "NDCG": ndcg_at_k(recs, relevant, k)})
    result = pd.DataFrame(rows, columns=["user_id", "Precision", "Recall", "NDCG"])
    return {f"Precision@{k}": result.Precision.mean(), f"Recall@{k}": result.Recall.mean(),
            f"NDCG@{k}": result.NDCG.mean(), "Evaluated users": len(result)}

ranking_results = pd.DataFrame([{"Model": name, **evaluate_ranking(fn, TOP_K)}
                                for name, fn in recommenders.items()])
eligible_users = test.loc[test.relevant, "user_id"].nunique()
print(f"Ranking includes {eligible_users}/{test.user_id.nunique()} users; "
      f"{test.relevant.sum()} relevant held-out items.")
display(ranking_results.round(4))

Ranking includes 7/12 users; 9 relevant held-out items.


,Model,Precision@3,Recall@3,NDCG@3,Evaluated users
0,Popularity Baseline,0.0952,0.1429,0.0991,7
1,Content-Based,0.2381,0.5714,0.3644,7
2,Item-Based CF,0.0476,0.1429,0.0714,7


## Steps 16–18 – Coverage, novelty, and diversity
These metrics use recommendations for **all training users**, unlike ranking's eligible subset.
Coverage is unique recommended movies divided by the 20-movie catalogue.
Novelty is mean `-log2(training item count / training user count)` across recommendation
occurrences; unseen items use a count of 1 as in the lab. It is a popularity proxy, not a
measurement of whether a particular user has heard of a movie.
Diversity is mean within-list `1 - genre cosine similarity` across all pairs.
Higher values indicate broader exposure, less common items, and less similar genres respectively.

In [9]:
def catalogue_coverage(recommender, k=3):
    recommended = {m for user_id in train.user_id.unique() for m in recommender(user_id, k)}
    return len(recommended) / movies.movie_id.nunique()

pop_counts = train.movie_id.value_counts().to_dict()
num_users = train.user_id.nunique()

def novelty_of_item(movie_id):
    return float(-np.log2(pop_counts.get(movie_id, 1) / num_users))

def average_novelty(recommender, k=3):
    values = [novelty_of_item(m) for u in train.user_id.unique() for m in recommender(u, k)]
    return float(np.mean(values)) if values else np.nan

def list_diversity(movie_ids):
    sims = [genre_similarity.loc[a, b] for a, b in combinations(movie_ids, 2)]
    return float(1 - np.mean(sims)) if sims else 0.0

def average_diversity(recommender, k=3):
    return float(np.mean([list_diversity(recommender(u, k)) for u in train.user_id.unique()]))

coverage_results = pd.DataFrame([{"Model": name, "Coverage": catalogue_coverage(fn)} for name, fn in recommenders.items()])
novelty_results = pd.DataFrame([{"Model": name, "Novelty": average_novelty(fn)} for name, fn in recommenders.items()])
diversity_results = pd.DataFrame([{"Model": name, "Diversity": average_diversity(fn)} for name, fn in recommenders.items()])
display(coverage_results.round(4), novelty_results.round(4), diversity_results.round(4))

,Model,Coverage
0,Popularity Baseline,0.45
1,Content-Based,0.80
2,Item-Based CF,0.75


,Model,Novelty
0,Popularity Baseline,1.5304
1,Content-Based,1.5332
2,Item-Based CF,1.6280


,Model,Diversity
0,Popularity Baseline,0.9444
1,Content-Based,0.5071
2,Item-Based CF,0.7917


## Step 19 – Unified evaluation table
MAE/RMSE: lower is better. Other quality metrics: higher is better, but the business
objective determines the balance. Novelty is in bits; it does not share the 0–1 scale.

In [10]:
summary = (rating_results.merge(ranking_results, on="Model")
    .merge(coverage_results, on="Model").merge(novelty_results, on="Model")
    .merge(diversity_results, on="Model"))
display(summary.round(4))
ranking_winners = summary.loc[np.isclose(summary["NDCG@3"], summary["NDCG@3"].max()), "Model"].tolist()
print("Best ranking model(s) by NDCG@3:", ", ".join(ranking_winners))

,Model,MAE,RMSE,Precision@3,Recall@3,NDCG@3,Evaluated users,Coverage,Novelty,Diversity
0,Popularity Baseline,0.7523,0.9204,0.0952,0.1429,0.0991,7,0.45,1.5304,0.9444
1,Content-Based,0.7405,0.8496,0.2381,0.5714,0.3644,7,0.80,1.5332,0.5071
2,Item-Based CF,0.8584,1.0195,0.0476,0.1429,0.0714,7,0.75,1.6280,0.7917


Best ranking model(s) by NDCG@3: Content-Based


## Steps 20–21 – Interpret trade-offs and choose a model
Management wants relevant discovery without the same popular titles dominating.
Use NDCG@3 for relevance and placement, coverage for catalogue exposure, and novelty
for discovery. Inspect diversity as additional context. Do not combine these unscaled
metrics into an arbitrary single score. The evidence-based recommendation below weighs
these objectives and states the cost of the choice.

In [11]:
display(summary.set_index("Model")[["NDCG@3", "Coverage", "Novelty", "Diversity"]].round(4))
print("Compare accuracy and ranking winners separately; they answer different questions.")

,NDCG@3,Coverage,Novelty,Diversity
Model,,,,
Popularity Baseline,0.0991,0.45,1.5304,0.9444
Content-Based,0.3644,0.80,1.5332,0.5071
Item-Based CF,0.0714,0.75,1.6280,0.7917


Compare accuracy and ranking winners separately; they answer different questions.


### Answers 10–13 – Business recommendation
**10. Choose the content-based model as the initial model for this dataset.** It leads
the three tested models on NDCG@3 (0.364), Precision@3 (0.238), and Recall@3 (0.571),
while exposing 16/20 movies (80% catalogue coverage). Popularity exposes 9/20 (45%),
and CF exposes 15/20 (75%). This is a provisional offline recommendation.

**11. The main metrics are NDCG@3, coverage, and novelty.** Content improves relevance
and catalogue exposure. Its novelty is 1.533 bits versus popularity's 1.530, a negligible
difference: these results do **not** show a meaningful improvement in rarity. CF has
the highest novelty (1.628), but NDCG@3 is only 0.071. Coverage measures distinct titles,
not how evenly exposure is distributed, so it cannot alone prove popular titles stop dominating.

**12. Accept lower within-list genre diversity** (content 0.507, popularity 0.944,
CF 0.792), and slightly less novelty than CF, in exchange for stronger observed relevance
and wider catalogue coverage. A genre-diversifying reranker could be evaluated later;
it has not been implemented or measured here.

**13. One key limitation:** ranking uses only seven eligible users and nine relevant
held-out ratings. The results are too small to justify a production-wide superiority claim.
Repeated splits, exposure-concentration measurement, and an online experiment would
strengthen a future decision.

**Rating prediction:** Content also has the lowest MAE (0.740) and RMSE (0.850).
Its MAE advantage over popularity is just 0.0118; without uncertainty estimates this
is not a clear statistical win. CF is worse than the baseline on both rating metrics.

## Step 22 – Extension A: K = 1, 3, 5
Use the same trained models, candidate universe, relevance labels, and eligible users.
Recommendation lists are nested under the deterministic ordering. Recall cannot decrease
as K grows. Precision may rise or fall depending on whether additional positions add hits.
With only two held-out items per user, recall can saturate and precision is constrained
by incomplete relevance labels. NDCG uses an ideal ranking recalculated for each K.

In [12]:
extension_rows = []
for k in [1, 3, 5]:
    for name, fn in recommenders.items():
        result = evaluate_ranking(fn, k)
        extension_rows.append({"Model": name, "K": k, "Precision@K": result[f"Precision@{k}"],
            "Recall@K": result[f"Recall@{k}"], "NDCG@K": result[f"NDCG@{k}"]})
extension_results = pd.DataFrame(extension_rows)
display(extension_results.round(4))
for name in recommenders:
    rows = extension_results.loc[extension_results.Model == name].sort_values("K")
    print(name, "— precision:", " → ".join(f"{v:.3f}" for v in rows["Precision@K"]),
          "; recall:", " → ".join(f"{v:.3f}" for v in rows["Recall@K"]))

,Model,K,Precision@K,Recall@K,NDCG@K
0,Popularity Baseline,1,0.0000,0.0000,0.0000
1,Content-Based,1,0.1429,0.0714,0.1429
2,Item-Based CF,1,0.0000,0.0000,0.0000
3,Popularity Baseline,3,0.0952,0.1429,0.0991
4,Content-Based,3,0.2381,0.5714,0.3644
5,Item-Based CF,3,0.0476,0.1429,0.0714
6,Popularity Baseline,5,0.0571,0.1429,0.0991
7,Content-Based,5,0.2000,0.8571,0.4812
8,Item-Based CF,5,0.0571,0.2143,0.1092


Popularity Baseline — precision: 0.000 → 0.095 → 0.057 ; recall: 0.000 → 0.143 → 0.143
Content-Based — precision: 0.143 → 0.238 → 0.200 ; recall: 0.071 → 0.571 → 0.857
Item-Based CF — precision: 0.000 → 0.048 → 0.057 ; recall: 0.000 → 0.143 → 0.214


### Interpretation of the extension
For content recommendations, Precision@K rises from 0.143 at K=1 to 0.238 at K=3,
then falls to 0.200 at K=5. Recall rises from 0.071 to 0.571 to 0.857: longer lists
recover more relevant held-out movies, while the final two positions dilute precision.
Popularity recall stays at 0.143 between K=3 and K=5, so adding positions adds no hits
and precision falls from 0.095 to 0.057. CF recall rises from 0 to 0.143 to 0.214.
Precision does not necessarily decrease with K: an initially poor top position can be
followed by relevant results. All comparisons use the same seven eligible users.

## Reproducibility and checks
Restart the Python kernel and run all cells from top to bottom. No download or external
service is required. The checks below cover split integrity, recommendation eligibility,
metric definitions, finite predictions, and monotonic recall in the extension.

In [13]:
assert precision_recall_at_k([1, 2, 3], [1, 3], 3) == (2 / 3, 1.0)
assert np.isclose(ndcg_at_k([1, 3, 2], [1, 3], 3), 1.0)
assert ndcg_at_k([1, 2, 3], [4], 3) == 0.0
assert np.isnan(ndcg_at_k([1, 2, 3], [], 3))
assert np.isclose(list_diversity([1, 5]), 0.0)  # identical Action/Sci-Fi genres
assert np.isclose(list_diversity([1, 2]), 1.0)  # disjoint genres
for predictions in [baseline_test, content_test, cf_test]:
    assert np.isfinite(predictions.prediction).all()
    assert predictions.prediction.between(1, 5).all()
for name, fn in recommenders.items():
    for user_id in train.user_id.unique():
        recs = fn(user_id, 5)
        assert len(recs) == 5 and len(set(recs)) == 5
        assert set(recs).issubset(unseen_movies(user_id))
    recall_values = extension_results.loc[extension_results.Model == name].sort_values("K")["Recall@K"]
    assert (np.diff(recall_values) >= -1e-12).all()
assert np.isfinite(summary.select_dtypes("number").to_numpy()).all()
assert summary[["Precision@3", "Recall@3", "NDCG@3", "Coverage", "Diversity"]].ge(-1e-12).all().all()
assert summary[["Precision@3", "Recall@3", "NDCG@3", "Coverage", "Diversity"]].le(1 + 1e-12).all().all()
print("All checks passed.")

All checks passed.


## Limitations
1. Only 12 users, 20 movies, and 24 held-out ratings; one random split is unstable and
   does not establish statistical superiority. Repeat per-user splits and estimate uncertainty.
2. Unrated candidates have unknown relevance; counting them as non-hits can penalize
   useful discoveries. Offline results do not measure satisfaction or engagement.
3. Users with no relevant held-out ratings are excluded from ranking averages, which
   changes the evaluated population; coverage/novelty/diversity use all users.
4. Genre metadata is coarse. Raw-rating cosine CF is sensitive to sparse overlap,
   rating habits, and popularity. No cold-user or temporal evaluation was performed.
5. The random split does not establish future performance; timestamps are not validated
   real dates. No validation set or model tuning is included.
6. Definitions for Steps 1–11 are reconstructed because those instructions were missing.
   The original lab may produce different numerical results.